In [ ]:
from pathlib import Path
from netCDF4 import Dataset
import wrf
import xarray as xr
import numpy as np
import pandas as pd
import cartopy.crs as crs
import cartopy.feature as cfeature
from matplotlib.cm import get_cmap
import matplotlib.pyplot as plt

In [ ]:
d01data = Path.home() / "Projects/dyndowndata/Merbok/wrf_d01/"
d02data = Path.home() / "Projects/dyndowndata/Merbok/wrf_d02/"

### Plot wind speed and surface-level pressure for a test file

Let's look at a 12 km resolution file first.

In [ ]:
testfile = d01data / "wrfout_d01_2022-09-17_12:00:00"
ncfile = Dataset(testfile)

In [ ]:
slp = wrf.getvar(ncfile, "slp", timeidx=0)
wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=0)
lats, lons = wrf.latlon_coords(slp)
cart_proj = wrf.get_cartopy(slp)

In [ ]:
for idx in range(6):
    wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=idx)
    wind.sel(wspd_wdir='wspd').plot()
    plt.show()

In [ ]:
for idx in range(6):
    slp = wrf.getvar(ncfile, "slp", timeidx=idx)
    slp.plot(levels=15)
    plt.show()

Now for a 4 km resolution file.

In [ ]:
testfile = d02data / "wrfout_d02_2022-09-17_12:00:00"
ncfile = Dataset(testfile)
slp = wrf.getvar(ncfile, "slp", timeidx=2)
slp = wrf.smooth2d(slp, 3, cenweight=2)
slp = slp.where(slp.XLONG < 0.0)
wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=2).sel(wspd_wdir='wspd')
wind = wind.where(wind.XLONG < 0.0)

Fancier plotting with cartopy.

In [ ]:
levels = np.arange(940, 1040, 10)
windlevels = np.arange(0, 28, 2)
levels

In [ ]:
projection = crs.AlbersEqualArea(
    central_longitude=-154.0, central_latitude=50.0, 
    standard_parallels=(55.0, 65.0))


fig = plt.figure(figsize=(15,10))
ax = plt.axes(projection=projection)
# ax.set_extent([-170, -150, 52.5, 72])
ax.set_extent([-170, -135, 55, 72])
ax.coastlines(color='#111', lw=1.5)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.add_feature(cfeature.RIVERS, color='blue')
wind.plot(
    transform=crs.PlateCarree(),
    x="XLONG", y="XLAT",
    ax=ax,
    cmap='gist_stern_r',
    # cmap='cubehelix',
    levels=windlevels,
)
contourplot = slp.plot.contour(
    transform=crs.PlateCarree(),
    x="XLONG", y="XLAT",
    ax=ax,
    levels=levels,
    colors="lavender",
    linewidth=0.75,
    kwargs=dict(inline=True)
)
ax.clabel(contourplot)

plt.title(f"Merbok SLP and wind speed, {slp.Time.dt.strftime('%Y-%m-%d %H:%M').data} UTC")

In [ ]:
hooper_bay_lat = 61.5280
hooper_bay_lon = -166.1076

wrf.ll_to_xy(ncfile, hooper_bay_lat, hooper_bay_lon, as_int=True)

In [ ]:
def plot_merbok(slp=slp, wind=wind, 
                show_figs=True, save_figs=False, 
                infix=None, all_AK=False):

    projection = crs.AlbersEqualArea(
        central_longitude=-154.0, central_latitude=50.0, 
        standard_parallels=(52.0, 65.0))
    levels = np.arange(940, 1040, 10)
    windlevels = np.arange(0, 28, 2)
    fig = plt.figure(figsize=(15,10))
    ax = plt.axes(projection=projection)
    if all_AK:
        ax.set_extent([-170, -135, 55, 72])
    else:
        ax.set_extent([-170, -150, 52.5, 72])
    ax.coastlines(color='#111', lw=1.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.RIVERS, color='blue')
    wind.plot(
        transform=crs.PlateCarree(),
        x="XLONG", y="XLAT",
        ax=ax,
        cmap='gist_stern_r',
        levels=windlevels,
    )
    contourplot = slp.plot.contour(
        transform=crs.PlateCarree(),
        x="XLONG", y="XLAT",
        ax=ax,
        levels=levels,
        colors="lavender",
        linewidth=0.75,
        kwargs=dict(inline=True)
    )
    ax.clabel(contourplot)
    plt.title(f"Merbok SLP and wind speed, {slp.Time.dt.strftime('%Y-%m-%d %H:%M').data} UTC")
    if show_figs:
        plt.show()
    if save_figs:
        if infix is not None: infix = f"_{infix}"
        fig.savefig(f"Merbok{infix}_{slp.Time.dt.strftime('%Y%m%d_%H%M_UTC').data}.png", bbox_inches='tight')

In [ ]:
for testfile in sorted(list(d01data.glob("wrfout*2022-09-1[6, 7, 8]*"))):

    ncfile = Dataset(testfile)
    for idx in range(6):
        slp = wrf.getvar(ncfile, "slp", timeidx=idx)
        slp = wrf.smooth2d(slp, 3, cenweight=4)
        slp = slp.where(slp.XLONG < 0.0)
        wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=idx).sel(wspd_wdir='wspd')
        wind = wind.where(wind.XLONG < 0.0)
        plot_merbok(wind=wind, slp=slp, infix='d01', all_AK=True, save_figs=True)

In [ ]:
for testfile in sorted(list(d02data.glob("wrfout*2022-09-17*"))):

    ncfile = Dataset(testfile)
    for idx in range(6):
        slp = wrf.getvar(ncfile, "slp", timeidx=idx)
        slp = wrf.smooth2d(slp, 3, cenweight=4)
        slp = slp.where(slp.XLONG < 0.0)
        wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=idx).sel(wspd_wdir='wspd')
        wind = wind.where(wind.XLONG < 0.0)
        plot_merbok()

In [ ]:
wrflist = [
    Dataset(item) for item in sorted(list(d02data.glob("wrfout*2022-09-*")))
]
slp = wrf.getvar(wrflist, 'slp', timeidx=wrf.ALL_TIMES, method="cat")
xx, yy = wrf.ll_to_xy(wrflist[0], hooper_bay_lat, hooper_bay_lon, as_int=True).data
print(xx, yy, hooper_bay_lat, hooper_bay_lon)
slp_d02_hooperbay = slp.isel(west_east=xx, south_north=yy).to_dataframe()

In [ ]:
slp_d02_hooperbay.slp.plot()

In [ ]:
wrflist = [
    Dataset(item) for item in sorted(list(d01data.glob("wrfout*2022-09-1[6, 7, 8]*")))
]
slp = wrf.getvar(wrflist, 'slp', timeidx=wrf.ALL_TIMES, method="cat")
xx, yy = wrf.ll_to_xy(wrflist[0], hooper_bay_lat, hooper_bay_lon, as_int=True).data
slp_d01_hooperbay = slp.isel(west_east=xx, south_north=yy).to_dataframe()

In [ ]:
slp_d01_hooperbay.slp.plot()

In [ ]:
xx, yy

In [ ]:
testfile = d01data / "wrfout_d01_2022-09-17_12:00:00"
ncfile = Dataset(testfile)
wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=2).sel(wspd_wdir='wspd')
wind = wind.where(wind.XLONG < 0.0)

In [ ]:
projection = crs.AlbersEqualArea(
    central_longitude=-154.0, central_latitude=50.0, 
    standard_parallels=(55.0, 65.0))


fig = plt.figure(figsize=(15,10))
ax = plt.axes(projection=projection)
ax.set_extent([-170, -130, 52.5, 72])
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
wind.plot(
    transform=crs.PlateCarree(),
    x="XLONG", y="XLAT",
    ax=ax,
    cmap='viridis',
    # levels=15
    vmin=0,
    vmax=26
)
ax.clabel(contourplot)

plt.title("10m Wind Speed (m/s)")

In [ ]:
testfile = d02data / "wrfout_d02_2022-09-17_12:00:00"
ncfile = Dataset(testfile)
wind = wrf.getvar(ncfile, "wspd_wdir10", timeidx=2).sel(wspd_wdir='wspd')
wind = wind.where(wind.XLONG < 0.0)

In [ ]:
projection = crs.AlbersEqualArea(
    central_longitude=-154.0, central_latitude=50.0, 
    standard_parallels=(55.0, 65.0))


fig = plt.figure(figsize=(15,10))
ax = plt.axes(projection=projection)
ax.set_extent([-170, -130, 52.5, 72])
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
wind.plot(
    transform=crs.PlateCarree(),
    x="XLONG", y="XLAT",
    ax=ax,
    cmap='viridis',
    # levels=15
    vmin=0,
    vmax=26
)
ax.clabel(contourplot)

plt.title("10m Wind Speed (m/s)")

### Make animated GIF

In [ ]:
from PIL import Image
import glob

In [ ]:
# Create the frames
frames = []
imgs = sorted(list(glob.glob("Merbok_d01*.png")))
frames = [Image.open(img) for img in imgs]
 
# Save into a GIF file that loops forever
frames[0].save('Merbok_d01_2.gif', format='GIF',
               append_images=frames[1:],
               save_all=True,
               duration=300, loop=0)